In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from pathlib import Path

DATA_PATH = Path("../data/raw")

CSV_PATH = DATA_PATH / "PS_20174392719_1491204439457_log.csv"
PARQUET_PATH = DATA_PATH / "paysim.parquet"

In [2]:
CHUNK_SIZE = 100_000

writer = None

try:
    for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNK_SIZE):

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                PARQUET_PATH,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)

finally:
    if writer is not None:
        writer.close()

print("Conversion complete.")

Conversion complete.


In [3]:
csv_size = CSV_PATH.stat().st_size / 1024**2
parquet_size = PARQUET_PATH.stat().st_size / 1024**2

print(f"CSV size: {csv_size:.2f} MB")
print(f"Parquet size: {parquet_size:.2f} MB")
print(f"Size reduction: {1 - parquet_size / csv_size:.2%}")

CSV size: 470.67 MB
Parquet size: 294.02 MB
Size reduction: 37.53%


In [4]:
metadata = pq.read_metadata(PARQUET_PATH)

print(f"Rows: {metadata.num_rows:,}")
print(f"Columns: {metadata.num_columns}")

assert metadata.num_rows == 6_362_620
assert metadata.num_columns == 11

Rows: 6,362,620
Columns: 11
